<a href="https://colab.research.google.com/github/anamallin/proteomicaEPB/blob/main/Pratica_MOFA2_DIABLO_MB_LGG_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Integração multiômica: MB × LGG

**Prática curta com MOFA2 e DIABLO no R — Minicurso EPB 2026**

### Objetivos

- Usar **MOFA2** para descobrir fatores latentes sem informar o diagnóstico;
- Usar **DIABLO** para selecionar uma assinatura que discrimine MB e LGG;
- Comparar as features priorizadas pelos dois métodos.

### Dados

- 81 amostras LGG e 16 amostras MB;
- 5.000 genes de transcriptômica;
- 5.600 proteínas;
- amostras pareadas e sem valores ausentes.

> Para manter a prática rápida, serão usadas as 1.000 features mais variáveis de cada camada. Essa filtragem não utiliza o diagnóstico.

## Antes de começar

1. Abra o notebook no Google Colab com **runtime R**;
2. Envie pelo painel **Files/Arquivos**:
   - `RNA_MB_LGG.tsv`
   - `PROTEOMICA_MB_LGG.tsv`
   - `METADADOS_MB_LGG.tsv`
3. Execute as células em ordem.

Os arquivos no Colab são temporários. Baixe o `.zip` produzido ao final.

## 1. Instalar pacotes

In [2]:
options(repos = c(CRAN = "https://cloud.r-project.org"), Ncpus = 2)

cran_packages <- c("readr", "dplyr", "ggplot2", "tibble", "reticulate", "BiocManager")
cran_missing <- cran_packages[
  !vapply(cran_packages, requireNamespace, logical(1), quietly = TRUE)
]
if (length(cran_missing)) install.packages(cran_missing)

bioc_packages <- c("MOFA2", "mixOmics")
bioc_missing <- bioc_packages[
  !vapply(bioc_packages, requireNamespace, logical(1), quietly = TRUE)
]
if (length(bioc_missing)) {
  BiocManager::install(bioc_missing, ask = FALSE, update = FALSE)
}

# Backend Python utilizado pelo MOFA2.
system("python3 -m pip install -q mofapy2==0.7.3")
cat("Instalação concluída.\n")

Installing packages into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘RcppTOML’, ‘here’, ‘png’


'getOption("repos")' replaces Bioconductor standard repositories, see
'help("repositories", package = "BiocManager")' for details.
Replacement repositories:
    CRAN: https://cloud.r-project.org

Bioconductor version 3.23 (BiocManager 1.30.27), R 4.6.1 (2026-06-24)

Installing package(s) 'BiocVersion', 'MOFA2', 'mixOmics'

also installing the dependencies ‘formatR’, ‘biocmake’, ‘XVector’, ‘abind’, ‘RcppEigen’, ‘sitmo’, ‘filelock’, ‘lambda.r’, ‘futile.options’, ‘rhdf5filters’, ‘Rhdf5lib’, ‘plyr’, ‘SparseArray’, ‘h5mread’, ‘BiocGenerics’, ‘S4Vectors’, ‘IRanges’, ‘S4Arrays’, ‘MatrixGenerics’, ‘FNN’, ‘irlba’, ‘RcppAnnoy’, ‘RSpectra’, ‘dqrng’, ‘RcppProgress’, ‘dir.expiry’, ‘futile.logger’, ‘snow’, ‘BH’, ‘rhdf5’, ‘reshape2’, ‘pheatmap’, ‘cowplot’, ‘ggrepel’, ‘HDF5Array’, ‘corrplot’, ‘DelayedArray’, ‘Rtsne’, ‘uwot’, ‘basilisk’, ‘igraph’, ‘ellipse’, ‘

Instalação concluída.


## 2. Carregar pacotes e configurar o modo da aula

In [1]:
suppressPackageStartupMessages({
  library(readr)
  library(dplyr)
  library(ggplot2)
  library(tibble)
  library(reticulate)
  library(MOFA2)
  library(mixOmics)
})

set.seed(2026)

# TRUE: tuning e validação mais rápidos para demonstração.
fast_mode <- TRUE
n_features_por_view <- 1000

output_dir <- "/content/resultados_MOFA_DIABLO_MB_LGG"
dir.create(output_dir, recursive = TRUE, showWarnings = FALSE)

files <- c(
  RNA = "/content/RNA_MB_LGG.tsv",
  Proteomica = "/content/PROTEOMICA_MB_LGG.tsv",
  Metadata = "/content/METADADOS_MB_LGG.tsv"
)

missing_files <- files[!file.exists(files)]
if (length(missing_files)) {
  stop(
    "Envie os arquivos pelo painel Files/Arquivos: ",
    paste(basename(missing_files), collapse = ", ")
  )
}

cat(R.version.string, "\n")
cat("Modo rápido:", fast_mode, "\n")

ERROR: Error in library(reticulate): there is no package called ‘reticulate’


## 3. Carregar e validar as matrizes

In [ ]:
read_feature_matrix <- function(path) {
  tab <- read_tsv(path, show_col_types = FALSE)
  feature_id <- tab[[1]]
  mat <- as.matrix(tab[, -1])
  rownames(mat) <- feature_id
  storage.mode(mat) <- "numeric"
  mat
}

rna_full <- read_feature_matrix(files[["RNA"]])
proteomica_full <- read_feature_matrix(files[["Proteomica"]])
metadata <- read_tsv(files[["Metadata"]], show_col_types = FALSE)
metadata$Diagnosis <- factor(metadata$Diagnosis, levels = c("LGG", "MB"))

stopifnot(
  identical(colnames(rna_full), colnames(proteomica_full)),
  identical(colnames(rna_full), metadata$Sample),
  sum(is.na(rna_full)) == 0,
  sum(is.na(proteomica_full)) == 0
)

cat("RNA:", nrow(rna_full), "features x", ncol(rna_full), "amostras\n")
cat("Proteômica:", nrow(proteomica_full), "features x", ncol(proteomica_full), "amostras\n")
print(table(metadata$Diagnosis))

## 4. Selecionar features variáveis para acelerar

In [ ]:
select_top_variable <- function(mat, n = 1000) {
  feature_sd <- apply(mat, 1, sd)
  keep <- order(feature_sd, decreasing = TRUE)[seq_len(min(n, nrow(mat)))]
  mat[keep, , drop = FALSE]
}

rna <- select_top_variable(rna_full, n_features_por_view)
proteomica <- select_top_variable(proteomica_full, n_features_por_view)

mofa_views <- list(
  Transcriptomica = rna,
  Proteomica = proteomica
)

cat("Features usadas em RNA:", nrow(rna), "\n")
cat("Features usadas em proteômica:", nrow(proteomica), "\n")
cat("Amostras:", ncol(rna), "\n")

### Pergunta

Por que a seleção por variância é aceitável antes de uma análise não supervisionada, enquanto selecionar features usando MB/LGG antes da validação causaria vazamento de informação?

# Parte A — MOFA2

MOFA2 aprende fatores latentes a partir das duas camadas **sem receber o diagnóstico**. Para interpretar um fator, observaremos:

- variância explicada em cada view;
- escores das amostras;
- pesos das features.

## 5. Criar e treinar o MOFA2

In [ ]:
python_candidates <- unique(c(Sys.which("python3"), Sys.which("python")))
python_candidates <- python_candidates[nzchar(python_candidates)]
python_ok <- vapply(python_candidates, function(path) {
  isTRUE(tryCatch({
    suppressWarnings(system2(path, "--version", stdout = FALSE, stderr = FALSE)) == 0
  }, error = function(e) FALSE))
}, logical(1))
python_bin <- python_candidates[python_ok][1]
if (is.na(python_bin)) stop("Python funcional não encontrado.")
reticulate::use_python(python_bin, required = TRUE)

mofa <- create_mofa(mofa_views)

model_opts <- get_default_model_options(mofa)
model_opts$num_factors <- 3

train_opts <- get_default_training_options(mofa)
train_opts$convergence_mode <- "fast"
train_opts$seed <- 2026

mofa <- prepare_mofa(
  mofa,
  model_options = model_opts,
  training_options = train_opts
)

mofa <- run_mofa(
  mofa,
  outfile = file.path(output_dir, "modelo_MOFA2_MB_LGG.hdf5"),
  use_basilisk = FALSE
)

cat("MOFA2 treinado.\n")

## 6. Variância explicada e escores

In [ ]:
plot_variance_explained(mofa, max_r2 = 20)

factors <- get_factors(mofa, factors = "all")[[1]]
factor_df <- as.data.frame(factors) %>%
  rownames_to_column("Sample") %>%
  left_join(metadata, by = "Sample")

p_factor <- ggplot(
  factor_df,
  aes(Factor1, Factor2, color = Diagnosis, label = Sample)
) +
  geom_point(size = 3, alpha = 0.85) +
  scale_color_manual(values = c(LGG = "#629BD0", MB = "#E7A8C6")) +
  labs(title = "Escores dos fatores MOFA2") +
  theme_minimal()

print(p_factor)
ggsave(
  file.path(output_dir, "MOFA2_fatores_MB_LGG.png"),
  p_factor, width = 8, height = 6, dpi = 200
)
write_csv(factor_df, file.path(output_dir, "MOFA2_escores.csv"))

### Interprete

1. Qual fator explica mais variação?
2. O fator separa MB e LGG mesmo sem usar o diagnóstico?
3. Existe alguma amostra distante das demais?

## 7. Features com maiores pesos no Fator 1

In [ ]:
weights_f1 <- get_weights(mofa, views = "all", factors = 1)

top_mofa <- lapply(weights_f1, function(w) {
  tibble(
    Feature = rownames(w),
    Weight = w[, 1],
    AbsWeight = abs(w[, 1])
  ) %>%
    arrange(desc(AbsWeight)) %>%
    slice_head(n = 20)
})

cat("Top RNA — Fator 1\n")
print(top_mofa$Transcriptomica)
cat("\nTop proteínas — Fator 1\n")
print(top_mofa$Proteomica)

write_csv(
  top_mofa$Transcriptomica,
  file.path(output_dir, "MOFA2_top20_RNA_Fator1.csv")
)
write_csv(
  top_mofa$Proteomica,
  file.path(output_dir, "MOFA2_top20_PROTEO_Fator1.csv")
)

# Parte B — DIABLO

DIABLO usa o diagnóstico MB/LGG para selecionar features que simultaneamente:

- discriminam as classes;
- mantêm relação entre as camadas.

Como há desbalanceamento entre LGG e MB, observaremos principalmente o **BER — Balanced Error Rate**.

## 8. Preparar os blocos e ajustar `keepX`

In [ ]:
# mixOmics usa amostras nas linhas e features nas colunas.
X <- list(
  Transcriptomica = t(rna),
  Proteomica = t(proteomica)
)
Y <- metadata$Diagnosis

design <- matrix(
  c(0, 0.1,
    0.1, 0),
  nrow = 2,
  dimnames = list(names(X), names(X))
)

test_keepX <- list(
  Transcriptomica = c(10, 25, 50),
  Proteomica = c(10, 25, 50)
)

nrepeat_tuning <- if (fast_mode) 2 else 10

set.seed(2026)
tuned <- tune.block.splsda(
  X = X,
  Y = Y,
  ncomp = 2,
  test.keepX = test_keepX,
  design = design,
  validation = "Mfold",
  folds = 5,
  nrepeat = nrepeat_tuning,
  dist = "centroids.dist",
  progressBar = TRUE
)

keepX_final <- tuned$choice.keepX
print(keepX_final)

### Interprete

`keepX` é o número de features mantidas por camada e componente. Valores muito altos aumentam flexibilidade, mas podem reduzir estabilidade e interpretação.

## 9. Ajustar e visualizar o DIABLO

In [ ]:
diablo <- block.splsda(
  X = X,
  Y = Y,
  ncomp = 2,
  keepX = keepX_final,
  design = design
)

plotIndiv(
  diablo,
  group = Y,
  ind.names = FALSE,
  legend = TRUE,
  title = "DIABLO: MB versus LGG"
)

plotDiablo(diablo, ncomp = 1)

saveRDS(diablo, file.path(output_dir, "modelo_DIABLO_MB_LGG.rds"))

## 10. Features selecionadas

In [ ]:
selected_comp1 <- selectVar(diablo, comp = 1)

selected_rna <- tibble(
  Feature = selected_comp1$Transcriptomica$name,
  Loading = selected_comp1$Transcriptomica$value[, 1],
  View = "Transcriptomica"
)

selected_proteo <- tibble(
  Feature = selected_comp1$Proteomica$name,
  Loading = selected_comp1$Proteomica$value[, 1],
  View = "Proteomica"
)

selected_features <- bind_rows(selected_rna, selected_proteo)
print(selected_features)

write_csv(
  selected_features,
  file.path(output_dir, "DIABLO_features_componente1.csv")
)

circosPlot(diablo, cutoff = 0.7, line = TRUE)

## 11. Avaliar o desempenho

In [ ]:
nrepeat_perf <- if (fast_mode) 3 else 10

set.seed(2026)
perf_diablo <- perf(
  diablo,
  validation = "Mfold",
  folds = 5,
  nrepeat = nrepeat_perf,
  dist = "centroids.dist",
  progressBar = TRUE
)

cat("Taxas de erro e BER:\n")
print(perf_diablo$error.rate)

saveRDS(
  perf_diablo,
  file.path(output_dir, "DIABLO_validacao.rds")
)

### Interprete

1. O erro balanceado é aceitável para as duas classes?
2. O desempenho é semelhante entre componentes?
3. As features selecionadas permanecem estáveis ao repetir os folds?

> Uma separação visual perfeita no conjunto de treinamento não demonstra generalização.

# Parte C — Comparar MOFA2 e DIABLO

## 12. As duas abordagens priorizam as mesmas features?

In [ ]:
top20_mofa_rna <- top_mofa$Transcriptomica$Feature
top20_mofa_proteo <- top_mofa$Proteomica$Feature

comparison <- tibble(
  View = c("Transcriptomica", "Proteomica"),
  N_DIABLO = c(nrow(selected_rna), nrow(selected_proteo)),
  Em_comum_com_top20_MOFA = c(
    sum(selected_rna$Feature %in% top20_mofa_rna),
    sum(selected_proteo$Feature %in% top20_mofa_proteo)
  )
)

print(comparison)
write_csv(comparison, file.path(output_dir, "comparacao_MOFA2_DIABLO.csv"))

### Síntese

- **MOFA2:** quais eixos organizam a variação das amostras?
- **DIABLO:** quais features discriminam MB e LGG?
- Concordância entre métodos fortalece uma hipótese, mas não prova causalidade.
- Features exclusivas também podem ser informativas, pois os métodos respondem a perguntas diferentes.

## 13. Compactar os resultados

In [ ]:
summary_lines <- c(
  "PRÁTICA MOFA2 E DIABLO — MB VERSUS LGG",
  sprintf("Amostras: %d", nrow(metadata)),
  sprintf("LGG: %d", sum(metadata$Diagnosis == "LGG")),
  sprintf("MB: %d", sum(metadata$Diagnosis == "MB")),
  sprintf("Features RNA usadas: %d", nrow(rna)),
  sprintf("Features proteômicas usadas: %d", nrow(proteomica)),
  sprintf("Modo rápido: %s", fast_mode)
)
writeLines(summary_lines, file.path(output_dir, "RESUMO.txt"))
cat(paste(summary_lines, collapse = "\n"), "\n")

zip_file <- "/content/resultados_MOFA_DIABLO_MB_LGG.zip"
if (file.exists(zip_file)) file.remove(zip_file)

old_wd <- getwd()
setwd("/content")
system("zip -rq resultados_MOFA_DIABLO_MB_LGG.zip resultados_MOFA_DIABLO_MB_LGG")
setwd(old_wd)

cat("\nArquivo gerado:", zip_file, "\n")
cat("Baixe pelo painel Files/Arquivos do Colab.\n")

## Checklist final

- O MOFA2 encontrou um fator associado a MB/LGG sem receber o diagnóstico?
- Esse fator explica RNA, proteína ou ambas as views?
- O DIABLO apresentou BER adequado, considerando o desbalanceamento?
- Quais features aparecem nos dois métodos?
- Como você validaria essa assinatura em uma coorte independente?